## 2 weeks 

In [1]:
from functools import wraps

import torch
import torch.nn as nn
from typing import Tuple, Union, Optional, List, Any
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np

In [10]:
# a utility for calculating running average
class AverageMeter:
    def __init__(self):
        self.num = 0
        self.tot = 0

    def update(self, val: float, sz: float):
        self.num += val * sz
        self.tot += sz

    def calculate(self) -> float:
        return self.num / self.tot


def averager(func):
    num = 0
    total = 0
    # val = 0.0

    @wraps(func)
    def inner(val, sz):
        nonlocal num
        nonlocal total
        num += val * sz
        total += sz
        print(num, total, num / total)
        return num, total, num / total

    return inner

In [11]:
@averager
def calc_product(n, a):
    return n, a


print(calc_product(2, 3))

6 3 2.0
(6, 3, 2.0)


# Problem 2: Implement a Transformer

## Part 2.A

In [81]:
x = torch.randn(3, 32, 32)
n = len(x) - 1
print(x[0, :, 1])

tensor([ 0.0132,  0.0141, -0.2192,  0.7527,  0.7450, -1.6731, -2.2302,  1.2135,
         0.0835, -0.0712, -1.8970, -0.8181,  2.0153, -0.8518,  0.2404,  0.0372,
         1.3849, -2.6113,  0.2380, -0.6576, -0.9269,  0.2252, -1.5828,  0.6593,
        -1.0152,  1.4961,  2.1616, -1.0257, -0.3284,  0.7789, -0.4451, -1.2559])


In [27]:
x.shape

torch.Size([3, 32, 32])

In [37]:
x1 = torch.randn(32, 32)

In [57]:
in_features = 32
out_features = 1
model = nn.Linear(in_features, out_features, bias=False)

In [58]:
model

Linear(in_features=32, out_features=1, bias=False)

In [59]:
model(x).shape

torch.Size([3, 32, 1])

In [60]:
model(x1).mean()

tensor(-0.0383, grad_fn=<MeanBackward0>)

In [61]:
model.weight.grad

In [62]:
model.weight.requires_grad

True

In [63]:
model.weight

Parameter containing:
tensor([[ 0.1229, -0.0351, -0.1665,  0.1563, -0.0743,  0.0147,  0.0234,  0.1186,
          0.1270, -0.1536, -0.1654,  0.1441,  0.1344,  0.1626, -0.0675, -0.0767,
          0.0570,  0.0046,  0.0925,  0.1335, -0.1738, -0.0198,  0.0640, -0.0869,
         -0.0735, -0.1098,  0.1605, -0.0715, -0.0976, -0.1375,  0.1040, -0.0107]],
       requires_grad=True)

In [64]:
print(model.weight.grad)

None


In [65]:
model.weight.requires_grad_(False)

Parameter containing:
tensor([[ 0.1229, -0.0351, -0.1665,  0.1563, -0.0743,  0.0147,  0.0234,  0.1186,
          0.1270, -0.1536, -0.1654,  0.1441,  0.1344,  0.1626, -0.0675, -0.0767,
          0.0570,  0.0046,  0.0925,  0.1335, -0.1738, -0.0198,  0.0640, -0.0869,
         -0.0735, -0.1098,  0.1605, -0.0715, -0.0976, -0.1375,  0.1040, -0.0107]])

In [66]:
Q = torch.randn(32, 100, 10)
K = torch.randn(32, 100, 10)
V = torch.randn(32, 100, 10)

In [67]:
(Q @ K.transpose(1, -1)).shape

torch.Size([32, 100, 100])

In [68]:
(torch.matmul(Q, K.transpose(dim0=1, dim1=2)) @ V).shape

torch.Size([32, 100, 10])

In [69]:
# torch.softmax()

## Attention Head

In [257]:
import math
import copy


class AttentionHead(nn.Module):
    def __init__(self, dim: int, n_hidden: int):
        # dim: the dimension of the input
        # n_hidden: the dimension of the keys, queries, and values

        super().__init__()

        self.W_K = nn.Linear(dim, n_hidden)  # W_K weight matrix
        self.W_Q = nn.Linear(dim, n_hidden)  # W_Q weight matrix
        self.W_V = nn.Linear(dim, n_hidden)  # W_V weight matrix
        self.n_hidden = n_hidden
        self.dim = dim
        # self.temp = torch.tensor([])
        # self.temp2 = torch.tensor([])
        # self.temp3 = torch.tensor([])

        print(self.count_params())
        # self.calculate_shapes()

    def count_params(self):
        return sum(p.view(-1).shape[0] for p in self.parameters())

    # def calculate_shapes(self):
    #     print("Attention Matrix shape",self.temp.shape)
    #     print(f"Before Normalizing Attention Head: {self.temp2.shape}")
    #     print(f"Attention Head final shape: {self.temp3.shape}")

    # todo
    def calc_attention_mask(self, inp, mask: Optional[torch.Tensor] = None):
        if not mask:
            return inp
        else:
            first = mask[0, :, 1]
            second = mask[0, :, 2]
            for i in first:
                for j in second:
                    if j == 0:
                        inp = -inp
            return inp

    # def forward(
    #     self, x: torch.Tensor, attn_mask: Optional[torch.Tensor]=None
    # ) -> Tuple[torch.Tensor, torch.Tensor]:
    #     """
    #     # x                the inputs. shape: (B x T x dim)
    #     # attn_mask        an attention mask. If None, ignore. If not None, then mask[b, i, j]
    #     #                  contains 1 if (in batch b) token i should attend on token j and 0
    #     #                  otherwise. shape: (B x T x T)
    #     #
    #     # Outputs:
    #     # attn_output      the output of performing self-attention on x. shape: (Batch x Num_tokens x n_hidden)
    #     # alpha            the attention weights (after softmax). shape: (B x T x T)
    #     #
    #     """
    #     print(f"Attention Mask:{attn_mask}")
    #
    #     A, Z = None, None
    #     """
    #     #todo : Compute self attention on x.
    #     #       (1) First project x to the query Q, key K, value V.
    #     #       (2) Then compute the attention weights alpha as:
    #     #                  alpha = softmax(QK^T/sqrt(n_hidden))
    #     #           Make sure to take into account attn_mask such that token i does not attend on token
    #     #           j if attn_mask[b, i, j] == 0. (Hint, in such a case, what value should you set the weight
    #     #           to before the softmax so that after the softmax the value is 0?)
    #     #       (3) The output is a linear combination of the values (weighted by the alphas):
    #     #                  out = alpha V
    #     #       (4) return the output and the alpha after the softmax
    #
    #     # ======= Answer START ========
    #     """
    #     print(f"x={type(x)},{x.shape if type(x) is torch.tensor else len(x)} in attention head")
    #     Q = self.W_Q(x).T
    #     K = self.W_K(x).T
    #     V = self.W_V(x).T
    #     print(f"Without Transpose:{self.W_Q(x).shape}")
    #     print(f"{Q.shape=},{K.shape=},{V.shape=}")
    #     #print(f"{(Q@K.T).shape=}\n\n\n-------------")
    #     #print(f"{torch.matmul(Q,K.T).shape}")
    #     A = torch.matmul(Q,K.transpose(dim0=1,dim1=2))
    #     print(f"{A.shape=}")
    #     self.temp = A
    #     #print(A.shape) # dim*dim
    #     #todo -> causal masking auto regressive.. for time based
    #     # if not attn_mask:
    #     #     out = alpha
    #     # else:
    #     #     first = attn_mask[0,:,1]
    #     #     second = attn_mask[0,:,2]
    #     #     for i in first:
    #     #         for j in second:
    #     #             if j==0:
    #     #                 out=-alpha
    #
    #     #mask = self.calc_attention_mask(A,attn_mask)
    #     #print(self.calculate_shapes())
    #     if attn_mask is not None:
    #         Z=A+attn_mask
    #     else:
    #         Z=A
    #     self.temp2 = Z
    #     #print(f"Masking score: {inp.shape}")
    #
    #     Z = torch.softmax(Z/n_hidden**.5,dim=-1)
    #     print(f"before multiplication{Z.shape=}")
    #     Z = torch.matmul(Z,V)
    #     #print(f"Checking shape along dim 1: {torch.softmax(Z,dim=1).shape}")
    #     print(f"{Z.shape=}")
    #
    #     self.temp3 = Z
    #     #self.calculate_shapes()
    #
    #
    #
    #
    #
    #
    #     #attn_output = torch.softmax(attn_scores,dim=-1)
    #
    #     # ======= Answer  END ========
    #     print("Attention Head Successful!")
    #     print(f"Attention output shape={A.shape}, Z={Z.shape} calculated for attention head\n\n------------")
    #
    #     return A,Z

    def forward(
        self, x: torch.Tensor, attn_mask: Optional[torch.Tensor] = None
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        # x shape: (B, T, dim)
        print("\n\n-----------------------\nIn Attention Head:\n\n=")
        print(f"{x.shape=}")
        # 1. Project inputs to Q, K, V (Keep 3D structure intact)
        Q = self.W_Q(x)  # (B, T, n_hidden)
        K = self.W_K(x)  # (B, T, n_hidden)
        V = self.W_V(x)  # (B, T, n_hidden)
        # print(f"{Q.shape=},{K.shape=},{V.shape=}")
        # 2. Compute attention scores matrix
        # Transpose the last two dimensions of K to match (B, n_hidden, T)
        A = torch.matmul(Q, K.transpose(-2, -1))  # Shape: (B, T, T)
        # print(f"{A.shape=}")
        scores = A
        A = Q @ V.transpose(-2, -1)
        # print(f"Second way {A.shape=}")
        # 3. Apply the Attention Mask safely before Softmax
        #todo
        if attn_mask is not None:
            # Everywhere mask is 0, fill scores with a massive negative number
            A = A.masked_fill(attn_mask == 0, float("-inf"))

        # 4. Normalize and apply Softmax to get Alpha weights
        #doing layer by layer so in each layer n_hidden dimensions
        A = torch.softmax(A / math.sqrt(self.n_hidden), dim=-1)  # Shape: (B, T, T)

        # 5. Compute the final context vector
        # (B, T, T) x (B, T, n_hidden) -> (B, T, n_hidden)
        Z = A @ V

        # 6. Return strictly in the order requested by your docstring/parent class
        return Z, A

    # def backward(self):

### Debug

In [82]:
import torch
import torch.nn as nn

# 1. Define the dimensions
BATCH_SIZE = 10
SEQ_LEN = 100
INPUT_DIM = 64
OUTPUT_DIM = 32

# 2. Instantiate the linear layer
# It expects the incoming features to be 64, and transforms them into 32
linear_layer = nn.Sequential(
    nn.Linear(in_features=INPUT_DIM, out_features=1),
    nn.Linear(in_features=1, out_features=OUTPUT_DIM),
)

# 3. Create a 3D input tensor matching your Transformer data layout
# Shape: (Batch Size, Sequence Length, Features)
x = torch.randn(BATCH_SIZE, SEQ_LEN, INPUT_DIM)
print(f"Input shape:  {x.shape}")  # Output: torch.Size([10, 100, 64])

# 4. Pass the tensor through the layer
output = linear_layer(x)
print(f"Output shape: {output.shape}")  # Output: torch.Size([10, 100, 32])

Input shape:  torch.Size([10, 100, 64])
Output shape: torch.Size([10, 100, 32])


In [86]:
x.transpose(-2, -1).shape

torch.Size([10, 64, 100])

In [89]:
attention = AttentionHead(INPUT_DIM, OUTPUT_DIM)
attention.count_params()
Z, A = attention(x)

6240


-----------------------
In Attention Head:

=
x.shape=torch.Size([10, 100, 64])
Q.shape=torch.Size([10, 100, 32]),K.shape=torch.Size([10, 100, 32]),V.shape=torch.Size([10, 100, 32])
A.shape=torch.Size([10, 100, 100])
Second way A.shape=torch.Size([10, 100, 100])


In [90]:
Z.shape

torch.Size([10, 100, 32])

In [91]:
A.shape

torch.Size([10, 100, 100])

In [289]:
x = torch.tensor([[2.0, 3]] * 3)
x

tensor([[2., 3.],
        [2., 3.],
        [2., 3.]])

In [290]:
inp = torch.randn(10, 100, 64)

In [291]:
attention_mask = torch.tensor([[0, 1.0], [1.0, 0]])

In [292]:
x.shape

torch.Size([3, 2])

In [293]:
# import copy
# attention_mask = copy.deepcopy(x)

In [294]:
# model(inp,attn_mask=None)

In [295]:
torch.tensor([[2.0, 3]] * 3) + torch.tensor([[2, 3]])

tensor([[4., 6.],
        [4., 6.],
        [4., 6.]])

In [296]:
# class LinearModules(nn.Module):
#     def __init__(self, dim,hidden,heads):
#         super().__init__()
#         self.linear_list = nn.ModuleList([
#            AttentionHead(dim,hidden) for _ in range(heads)
#         ])
#         #print(self.linear_list[0].count_params())
#
#     def forward(self,x,attn_mask=None):
#         print(f"x={x}length={len(x)} in Multi Attention Head\n\n------------")
#         for index,layer in enumerate(self.linear_list):
#             x=self.linear_list[index](x)
#             W0 = layer(x)
#             #print(f"W0={W0}")
#
#             #x=list(x)
#             print(f"x={x}len={len(x)},{type(x)}")
#
#         print(f"x={x}len={len(x)} in Multi Attention Head after")
#         print(f"Concated Heads: {torch.concat(x)}")
#         return torch.concat(x)@W0
#         #return x

In [297]:
# model = LinearModules(2,3,1)
# model

In [298]:
# for p in model.parameters():
#     print(p.shape)

In [299]:
for i, p in model.named_modules():
    print(i, p)

 AttentionHead(
  (W_K): Linear(in_features=2, out_features=2, bias=True)
  (W_Q): Linear(in_features=2, out_features=2, bias=True)
  (W_V): Linear(in_features=2, out_features=2, bias=True)
)
W_K Linear(in_features=2, out_features=2, bias=True)
W_Q Linear(in_features=2, out_features=2, bias=True)
W_V Linear(in_features=2, out_features=2, bias=True)


In [300]:
m = AttentionHead(2, 3)
m.train()
pred = m(torch.tensor([[2.0, 2]] * 3))

27


In [301]:
x, y = pred
x

tensor([[-0.1286,  0.4172,  0.5486],
        [-0.1286,  0.4172,  0.5486],
        [-0.1286,  0.4172,  0.5486]], grad_fn=<MmBackward0>)

In [302]:
print(y)

tensor([[0.3333, 0.3333, 0.3333],
        [0.3333, 0.3333, 0.3333],
        [0.3333, 0.3333, 0.3333]], grad_fn=<SoftmaxBackward0>)


In [303]:
# model.train()
# pred = model(torch.tensor([[2.0,2]]*3))
# pred

In [258]:
# list(model.get_submodule("linear_list.0").parameters())

## Part 2.B Multi Attention Head

### to do in parallel without loops

This line creates a standard linear layer (W_o) that merges the independent representations learned by your individual attention heads back into a single unified vector space.
Here is exactly why it is configured with those dimensions:


#### 1. Fixing the Dimension Explosion


Each individual AttentionHead outputs a tensor of shape (B, T, n_hidden).
When you loop through all your heads and concatenate (torch.cat) them along the final axis, their dimensions combine like this:
$$\text{Shape: } (B, T, \color{lightgreen}{n\_hidden}) \times \text{num\_heads} \xrightarrow{\text{torch.cat}} (B, T, \color{lightblue}{num\_heads \times n\_hidden})$$
If you did not use this projection layer, the data moving to the next block in your neural network would be far too wide (num_heads * n_hidden) instead of the standard model dimension (dim).


#### 2. Allowing Heads to Communicate

Concatenation simply glues the heads next to each other, meaning the features found by Head #1 do not interact with Head #2. Passing the concatenated tensor through nn.Linear performs a matrix multiplication that mathematically mixes all the heads' outputs together.


#### 3. Re-aligning with Residual Connections


In transformer architectures, you usually add the original input $x$ back to the attention output (a residual connection):
$$\text{Output} = x + \text{Attention}(x)$$
For this addition ($+$) to work, the final output must have the exact same shape as the input tensor $x$, which is (B, T, dim). The out_proj layer guarantees this shape matches perfectly.



In [259]:
class MultiHeadedAttention(nn.Module):
    def __init__(self, dim: int, n_hidden: int, num_heads: int):
        # dim: the dimension of the input
        # n_hidden: the hidden dimensions for the attention layer
        # num_heads: the number of attention heads
        super().__init__()
        self.H = num_heads
        # TODO: set up your parameters for multi-head attention. You should initialize
        #       num_heads attention heads (see nn.ModuleList) as well as a linear layer
        #       that projects the concatenated outputs of each head into dim
        #       (what size should this linear layer be?)

        # ======= Answer START ========
        # self.attention = AttentionHead(dim=dim,n_hidden=n_hidden)
        self.linear_list = nn.ModuleList(
            [AttentionHead(dim, n_hidden) for _ in range(num_heads)]
        )
        self.dim = dim

        # 2. Final linear projection layer
        # Maps concatenated outputs (num_heads * n_hidden) back to original dim
        self.W0 = nn.Linear(n_hidden * num_heads, dim)

        # ======= Answer  END ========

    # todo in parallel
    # def forward(
    #     self, x: torch.Tensor, attn_mask: Optional[torch.Tensor]
    # ) -> Tuple[torch.Tensor, torch.Tensor]:
    #     # x                the inputs. shape: (B x T x dim)
    #     # attn_mask        an attention mask. If None, ignore. If not None, then mask[b, i, j]
    #     #                  contains 1 if (in batch b) token i should attend on token j and 0
    #     #                  otherwise. shape: (B x T x T)
    #     #
    #     # Outputs:
    #     # attn_output      the output of performing multi-headed self-attention on x.
    #     #                  shape: (B x T x dim)
    #     # attn_alphas      the attention weights of each of the attention heads.
    #     #                  shape: (B x Num_heads x T x T)
    #
    #     A,Z,W0 = None, None,None
    #
    #     # TODO: Compute multi-headed attention. Loop through each of your attention heads
    #     #       and collect the outputs. Concatenate them together along the hidden dimension,
    #     #       and then project them back into the output dimension (dim). Return both
    #     #       the final attention outputs as well as the alphas from each head.
    #
    #     # ======= Answer START ========
    #     print(f"x={x}length={len(x)},type={type(x)} in Multi Attention Head\n\n------------")

    def forward(
        self, x: torch.Tensor, attn_mask: Optional[torch.Tensor]
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        Z_concat = []
        A_concat = []
        head_outputs = []
        head_alphas = []
        print(f"\n\n In Multi headed Attention: {x.shape=}")
        for head in self.linear_list:
            #     Z,A = head(x,attn_mask=attn_mask)
            #     Z_concat.append(Z)
            #     A_concat.append(A)
            # # 4. Concatenate outputs from all heads along the last (hidden) dimension
            # # Resulting shape: (B, T, num_heads * n_hidden)
            # A=torch.cat(A_concat,dim=-1)
            # Z=self.W0(A)
            # Ah=torch.stack(Z_concat,dim=1)
            # return Z,Ah
            out, alpha = head(x, attn_mask=attn_mask)
            print(f"Multi Headed Attention: {out.shape=},{alpha.shape=}")

            head_outputs.append(out)  # Each item shape: (B, T, n_hidden)
            head_alphas.append(alpha)  # Each item shape: (B, T, T)

        # 4. Concatenate outputs from all heads along the last (hidden) dimension
        # Resulting shape: (B, T, num_heads * n_hidden)
        #print(f"{len(head_alphas[0])},{len(head_outputs[0])}")
        concatenated_heads = torch.cat(head_outputs, dim=-1)

        # concatenated_heads=concatenated_heads.transpose(0,1)

        # 5. Project the concatenated representation back to 'dim'
        # Resulting shape: (B, T, dim)
        #print(f"Multi Headed Attention: {concatenated_heads.shape=}")
        attn_output = self.W0(concatenated_heads)
        # print(
        #     f"Multi Headed Attention: {attn_output.shape=} With output head successful\n\n-----------"
        # )

        # 6. Stack attention weights from all heads along a new head dimension
        # Resulting shape: (B, num_heads, T, T)

        attn_alphas = torch.stack(head_alphas, dim=1)
        print(f"{attn_alphas.shape=}")
        print("Multi Headed Attention Successful\n\n\n--------------------------")
        return attn_output, attn_alphas

### Debug

In [97]:
Z.shape

torch.Size([10, 100, 32])

In [111]:
print(torch.stack((Z, Z), dim=1).shape)
print(torch.stack((Z, Z), dim=-1).shape)

torch.Size([10, 2, 100, 32])
torch.Size([10, 100, 32, 2])


In [110]:
print(torch.cat([Z, Z], dim=0).shape)

print(torch.cat([Z, Z], dim=-1).shape)
print(torch.cat([Z, Z], dim=1).shape)

torch.Size([20, 100, 32])
torch.Size([10, 100, 64])
torch.Size([10, 200, 32])


In [109]:
print(torch.concat([Z, Z], dim=0).shape)

print(torch.concat([Z, Z], dim=1).shape)
print(torch.concat([Z, Z], dim=-1).shape)

torch.Size([20, 100, 32])
torch.Size([10, 200, 32])
torch.Size([10, 100, 64])


In [246]:
# multi_attn_head_model = MultiHeadedAttention(dim=3,n_hidden=n_hidden,num_heads=20)

In [247]:
# multi_attn_head_model(x,attn_mask=None)

## Part 2.C

### Feed Forward Network

In [260]:
class FFN(nn.Module):
    def __init__(self, dim: int, n_hidden: int):
        # dim       the dimension of the input
        # n_hidden  the width of the linear layer

        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(dim),
            nn.Linear(dim, n_hidden),
            nn.GELU(),
            nn.Linear(n_hidden, dim),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x         the input. shape: (B x T x dim)

        # Outputs:
        # out       the output of the feed-forward network: (B x T x dim)
       # print("\n\n\nIn FFN-------------------------")
        print(f"{x.shape=}")
        out = self.net(x)
        #print(f"Out={out.shape=}")
        return out

### Attention Residual

In [261]:
# these are already implemented for you!


class AttentionResidual(nn.Module):
    def __init__(self, dim: int, attn_dim: int, mlp_dim: int, num_heads: int):
        # dim       the dimension of the input
        # attn_dim  the hidden dimension of the attention layer
        # mlp_dim   the hidden layer of the FFN
        # num_heads the number of heads in the attention layer
        super().__init__()

        self.attn = MultiHeadedAttention(dim, attn_dim, num_heads)
        self.ffn = FFN(dim, mlp_dim)

    def forward(
        self, x: torch.Tensor, attn_mask: torch.Tensor
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        # x                the inputs. shape: (B x T x dim)
        # attn_mask        an attention mask. If None, ignore. If not None, then mask[b, i, j]
        #                  contains 1 if (in batch b) token i should attend on token j and 0
        #                  otherwise. shape: (B x T x T)
        #
        # Outputs:
        # attn_output      shape: (B x T x dim)
        # attn_alphas      the attention weights of each of the attention heads.
        #                  shape: (B x Num_heads x T x T)
        #print("\n\n--------- Attention Residual")
        Z, A = self.attn(x=x, attn_mask=attn_mask)
        # Z,A = attn_out,alphas

        # print("\n\n  In Attention Residual")
        # print(f"{Z.shape=}\t,{x.shape=}\t.{A.shape=}\n")
        x = Z + x
        x = self.ffn(x) + x
        #print(f"{x.shape=}\n\n--------------")
        return x, A

In [198]:
print(x.shape)
model_ffn = nn.Sequential(
    nn.LayerNorm(x.shape[-1], eps=1e-6),
    nn.Linear(64, 1000),
    nn.ReLU(),
    nn.Linear(1000, 64),
)
layer_norm = nn.LayerNorm(x.shape[-1])
out = layer_norm(x)

torch.Size([10, 100, 3])


In [199]:
out.shape

torch.Size([10, 100, 3])

In [200]:
model_ffn

Sequential(
  (0): LayerNorm((3,), eps=1e-06, elementwise_affine=True, bias=True)
  (1): Linear(in_features=64, out_features=1000, bias=True)
  (2): ReLU()
  (3): Linear(in_features=1000, out_features=64, bias=True)
)

In [ ]:
model_ffn(x).shape

In [202]:
for key, value in model_ffn.state_dict().items():
    print(key, value.shape)

0.weight torch.Size([3])
0.bias torch.Size([3])
1.weight torch.Size([1000, 64])
1.bias torch.Size([1000])
3.weight torch.Size([64, 1000])
3.bias torch.Size([64])


In [203]:
print(list(model_ffn.named_modules()))

[('', Sequential(
  (0): LayerNorm((3,), eps=1e-06, elementwise_affine=True, bias=True)
  (1): Linear(in_features=64, out_features=1000, bias=True)
  (2): ReLU()
  (3): Linear(in_features=1000, out_features=64, bias=True)
)), ('0', LayerNorm((3,), eps=1e-06, elementwise_affine=True, bias=True)), ('1', Linear(in_features=64, out_features=1000, bias=True)), ('2', ReLU()), ('3', Linear(in_features=1000, out_features=64, bias=True))]


In [204]:
device = "mps"

In [205]:
num_tokens = 100
batch_size = 10
dim = 3
num_layers = 4
num_heads = 2
attn_dim = 32
model = AttentionResidual(dim=dim, attn_dim=attn_dim, mlp_dim=dim, num_heads=num_heads)

384
384


In [206]:
model

AttentionResidual(
  (attn): MultiHeadedAttention(
    (linear_list): ModuleList(
      (0-1): 2 x AttentionHead(
        (W_K): Linear(in_features=3, out_features=32, bias=True)
        (W_Q): Linear(in_features=3, out_features=32, bias=True)
        (W_V): Linear(in_features=3, out_features=32, bias=True)
      )
    )
    (W0): Linear(in_features=64, out_features=3, bias=True)
  )
  (ffn): FFN(
    (net): Sequential(
      (0): LayerNorm((3,), eps=1e-05, elementwise_affine=True, bias=True)
      (1): Linear(in_features=3, out_features=3, bias=True)
      (2): GELU(approximate='none')
      (3): Linear(in_features=3, out_features=3, bias=True)
    )
  )
)

In [207]:
print(list(model.named_modules()))

[('', AttentionResidual(
  (attn): MultiHeadedAttention(
    (linear_list): ModuleList(
      (0-1): 2 x AttentionHead(
        (W_K): Linear(in_features=3, out_features=32, bias=True)
        (W_Q): Linear(in_features=3, out_features=32, bias=True)
        (W_V): Linear(in_features=3, out_features=32, bias=True)
      )
    )
    (W0): Linear(in_features=64, out_features=3, bias=True)
  )
  (ffn): FFN(
    (net): Sequential(
      (0): LayerNorm((3,), eps=1e-05, elementwise_affine=True, bias=True)
      (1): Linear(in_features=3, out_features=3, bias=True)
      (2): GELU(approximate='none')
      (3): Linear(in_features=3, out_features=3, bias=True)
    )
  )
)), ('attn', MultiHeadedAttention(
  (linear_list): ModuleList(
    (0-1): 2 x AttentionHead(
      (W_K): Linear(in_features=3, out_features=32, bias=True)
      (W_Q): Linear(in_features=3, out_features=32, bias=True)
      (W_V): Linear(in_features=3, out_features=32, bias=True)
    )
  )
  (W0): Linear(in_features=64, out_fe

In [209]:
# (Batch Size = 32, Sequence Length = 3, Dim = 3)
x = torch.randn(batch_size, num_tokens, dim)

x, A = model(x, attn_mask=None)
print(x.shape, A.shape)



--------- Attention Residual


 In Multi headed Attention: x.shape=torch.Size([10, 100, 3])


-----------------------
In Attention Head:

=
x.shape=torch.Size([10, 100, 3])
Q.shape=torch.Size([10, 100, 32]),K.shape=torch.Size([10, 100, 32]),V.shape=torch.Size([10, 100, 32])
A.shape=torch.Size([10, 100, 100])
Second way A.shape=torch.Size([10, 100, 100])
Multi Headed Attention: out.shape=torch.Size([10, 100, 32]),alpha.shape=torch.Size([10, 100, 100])


-----------------------
In Attention Head:

=
x.shape=torch.Size([10, 100, 3])
Q.shape=torch.Size([10, 100, 32]),K.shape=torch.Size([10, 100, 32]),V.shape=torch.Size([10, 100, 32])
A.shape=torch.Size([10, 100, 100])
Second way A.shape=torch.Size([10, 100, 100])
Multi Headed Attention: out.shape=torch.Size([10, 100, 32]),alpha.shape=torch.Size([10, 100, 100])
10,10
Multi Headed Attention: concatenated_heads.shape=torch.Size([10, 100, 64])
Multi Headed Attention: attn_output.shape=torch.Size([10, 100, 3]) With output head successful

---

In [210]:
print(list(model.named_modules()))

[('', AttentionResidual(
  (attn): MultiHeadedAttention(
    (linear_list): ModuleList(
      (0-1): 2 x AttentionHead(
        (W_K): Linear(in_features=3, out_features=32, bias=True)
        (W_Q): Linear(in_features=3, out_features=32, bias=True)
        (W_V): Linear(in_features=3, out_features=32, bias=True)
      )
    )
    (W0): Linear(in_features=64, out_features=3, bias=True)
  )
  (ffn): FFN(
    (net): Sequential(
      (0): LayerNorm((3,), eps=1e-05, elementwise_affine=True, bias=True)
      (1): Linear(in_features=3, out_features=3, bias=True)
      (2): GELU(approximate='none')
      (3): Linear(in_features=3, out_features=3, bias=True)
    )
  )
)), ('attn', MultiHeadedAttention(
  (linear_list): ModuleList(
    (0-1): 2 x AttentionHead(
      (W_K): Linear(in_features=3, out_features=32, bias=True)
      (W_Q): Linear(in_features=3, out_features=32, bias=True)
      (W_V): Linear(in_features=3, out_features=32, bias=True)
    )
  )
  (W0): Linear(in_features=64, out_fe

This slide illustrates four foundational concepts from Chapter 12 of Understanding Deep Learning that you are wiring together right now.
Looking closely at the diagrams, we can connect the visual math to the exact code logic you are writing:
1. Residual Connection (Top Left)
What the slide shows: The standard ResNet style where identity x skips around two layers to compute $\mathcal{F}(x) + x$.
Your UDL Variant: As you noted earlier, the UDL textbook uses a Pre-LN layout where the normalization step is moved inside $\mathcal{F}(x)$, turning the formula into:
$$x \leftarrow x + \text{Layer}(\text{LayerNorm}(x))$$ 
2. Masking (Top Right)
What the slide shows: A Causal Mask matrix used for auto-regressive decoding. The black upper-triangular region is filled with 0s (or negative infinities in code) to ensure token i cannot look at future token j.
The "A" Box: The white region represents valid historical tokens where attention alignment probabilities are allowed to compute normally.
3. Layer Normalization vs. BatchNorm (Middle Right)
BatchNorm: Normalizes features across the batch axis ($N_{\text{batch}}$). This is dangerous for text sequences because different sentences in a batch shouldn't dictate each other's feature scaling.
LayerNorm: Normalizes across the feature dimension for each individual token separately (the horizontal gray row). This leaves the batch dimension completely isolated, preserving the independent batch universe rules we discussed.
4. Positional Encoding / ViT Structure (Bottom Left)
What the slide shows: This specific graphic maps out a Vision Transformer (ViT) pipeline. It takes an image, splits it into sequential patches (labeled 1 through 9), flattens them via a Linear Projection layer, appends an Extra learnable [class] embedding, and adds positional order keys before feeding the 3D tensor sequence directly into your TransformerEncoder.

Are you working on generating the lower-triangular causal mask matrix (torch.tril) for your autoregressive tests, or are you ready to code the final linear projection head?

In [244]:
x = x.to(device)
model(x, attn_mask=None)



-------------------------Transformer: torch.Size([10, 100, 64])


--------- Attention Residual


 In Multi headed Attention: x.shape=torch.Size([10, 100, 64])


-----------------------
In Attention Head:

=
x.shape=torch.Size([10, 100, 64])
Q.shape=torch.Size([10, 100, 32]),K.shape=torch.Size([10, 100, 32]),V.shape=torch.Size([10, 100, 32])
A.shape=torch.Size([10, 100, 100])
Second way A.shape=torch.Size([10, 100, 100])
Multi Headed Attention: out.shape=torch.Size([10, 100, 32]),alpha.shape=torch.Size([10, 100, 100])


-----------------------
In Attention Head:

=
x.shape=torch.Size([10, 100, 64])
Q.shape=torch.Size([10, 100, 32]),K.shape=torch.Size([10, 100, 32]),V.shape=torch.Size([10, 100, 32])
A.shape=torch.Size([10, 100, 100])
Second way A.shape=torch.Size([10, 100, 100])
Multi Headed Attention: out.shape=torch.Size([10, 100, 32]),alpha.shape=torch.Size([10, 100, 100])
10,10
Multi Headed Attention: concatenated_heads.shape=torch.Size([10, 100, 64])
Multi Headed Attention: attn_o

(tensor([[[ 1.2018, -0.3040,  0.9147,  ...,  0.5046,  0.8949,  0.1112],
          [ 1.3725,  1.8342,  0.6562,  ...,  2.1940, -0.1255, -0.5848],
          [-1.7116, -1.4464, -1.3505,  ..., -0.7707, -1.3727, -0.3219],
          ...,
          [-1.3637, -1.7117,  0.0146,  ..., -0.8940,  0.4131, -0.1734],
          [ 0.4744,  0.8094, -0.5144,  ..., -0.3140, -1.7001, -1.1953],
          [ 2.0051,  1.2237,  1.0178,  ..., -0.6662,  1.7915, -0.1734]],
 
         [[ 1.6982,  2.2604,  2.6040,  ...,  1.2220,  0.1551, -2.4207],
          [-0.8604,  2.2699,  0.9729,  ...,  3.2493,  0.9646, -2.4283],
          [ 1.1511,  1.4828,  2.1003,  ...,  1.2170, -2.1161,  0.4873],
          ...,
          [ 0.7195,  0.2482, -0.7204,  ..., -0.7507, -2.7129, -1.3031],
          [ 2.1307,  2.0208, -0.0046,  ...,  2.1743, -2.0467, -1.4266],
          [ 2.0066,  1.1419,  2.1398,  ...,  0.0322, -1.1305, -2.1128]],
 
         [[-0.3314,  0.8366, -0.1122,  ...,  0.2865,  0.4501, -0.6297],
          [ 2.4667, -1.3822,

In [392]:
# model(x,attn_mask=None)

## Transformer
 The FFN performs
normalization along with linear layers interspersed by GELUs 2 (similar to the MLPs
you’ve seen in previous PSETs). AttentionResidual then sends the input through
both multiheaded attention and the FFN, adding a residual after each step.
Deliverable Implement Transformer, which passes the input through successive
AttentionResidual layers.

In [262]:
class Transformer(nn.Module):
    def __init__(
        self, dim: int, attn_dim: int, mlp_dim: int, num_heads: int, num_layers: int
    ):
        # dim       the dimension of the input
        # attn_dim  the hidden dimension of the attention layer
        # mlp_dim   the hidden layer of the FFN
        # num_heads the number of heads in the attention layer
        # num_layers the number of attention layers.
        super().__init__()


        # ======= Answer START ========
        self.layers = nn.ModuleList(
            [
                AttentionResidual(dim, attn_dim, mlp_dim, num_heads)
                for _ in range(num_layers)
            ]
        )

        # ======= Answer END ========

    def forward(
        self, x: torch.Tensor, attn_mask: torch.Tensor, return_attn=False
    ) -> Tuple[torch.Tensor, Optional[torch.Tensor]]:
        """
        # x                the inputs. shape: (B x T x dim)
        # attn_mask        an attention mask. Pass this to each of the AttentionResidual layers!
        #                  shape: (B x T x T)
        #
        # Outputs:
        # attn_output      shape: (B x T x dim)
        # attn_alphas      If return_attn is False, return None. Otherwise return the attention weights
        #                  of each of each of the attention heads for each of the layers.
        #                  shape: (B x Num_layers x Num_heads x T x T)
        """
        output = None
        collected_attns = []

        # TODO: Implement the transformer forward pass! Pass the input successively through each of the
        # AttentionResidual layers. If return_attn is True, collect the alphas along the way.

        # ======= Answer START ========
        print(f"\n\n-------------------------" f"Transformer: {x.shape}")
        for residual in self.layers:

            x, alphas = residual(x, attn_mask=attn_mask)
            # if return_attn is None:
            #     alphas = None
           # print(type(alphas), alphas.shape)
            if return_attn is not None:
                collected_attns.append(alphas)

        #print(x.shape)
        #print("return attention:", return_attn)
        # ======= Answer END ========
        if return_attn is False:
            return x, None
        else:
            return (
                x,
                torch.stack(collected_attns, dim=1),
            )
        # return x if not return_attn else x,collected_attns
        # return output, collected_attns

In [263]:
num_tokens = 100
batch_size = 10
dim = 64
num_layers = 4
num_heads = 2
attn_dim = 32
inp = torch.randn(batch_size, num_tokens, dim).to(device)
model = Transformer(
    dim=dim, attn_dim=32, mlp_dim=dim, num_heads=num_heads, num_layers=num_layers
).to(device)

6240
6240
6240
6240
6240
6240
6240
6240


In [264]:
out, a = model(inp, attn_mask=None, return_attn=False)



-------------------------Transformer: torch.Size([10, 100, 64])


 In Multi headed Attention: x.shape=torch.Size([10, 100, 64])


-----------------------
In Attention Head:

=
x.shape=torch.Size([10, 100, 64])
Multi Headed Attention: out.shape=torch.Size([10, 100, 32]),alpha.shape=torch.Size([10, 100, 100])


-----------------------
In Attention Head:

=
x.shape=torch.Size([10, 100, 64])
Multi Headed Attention: out.shape=torch.Size([10, 100, 32]),alpha.shape=torch.Size([10, 100, 100])
attn_alphas.shape=torch.Size([10, 2, 100, 100])
Multi Headed Attention Successful


--------------------------
x.shape=torch.Size([10, 100, 64])


 In Multi headed Attention: x.shape=torch.Size([10, 100, 64])


-----------------------
In Attention Head:

=
x.shape=torch.Size([10, 100, 64])
Multi Headed Attention: out.shape=torch.Size([10, 100, 32]),alpha.shape=torch.Size([10, 100, 100])


-----------------------
In Attention Head:

=
x.shape=torch.Size([10, 100, 64])
Multi Headed Attention: out.shape=to

In [245]:
out

tensor([[[ 0.7045,  1.3035, -0.0502,  ..., -0.9029, -2.0673,  0.4986],
         [ 0.0868, -0.1023, -0.6664,  ..., -0.3703,  0.7169,  1.4581],
         [ 0.9088, -0.5516, -0.0536,  ...,  0.7431, -1.4278,  0.1272],
         ...,
         [ 1.3990, -0.6850,  0.6995,  ...,  0.6742,  1.2438, -0.3035],
         [ 0.6918,  2.7398,  1.5379,  ..., -0.8242, -0.4836, -1.4246],
         [-0.3065,  3.3416, -0.6845,  ...,  0.1035,  1.9206,  1.5765]],

        [[ 0.3972,  1.5034,  1.2574,  ...,  1.7454,  1.0258, -0.6596],
         [ 1.4546,  2.5945,  0.5108,  ...,  2.1685,  1.2092, -0.4204],
         [ 0.7304,  0.2586, -1.9997,  ..., -0.3462, -0.9824,  0.5241],
         ...,
         [-0.1993,  0.5742,  0.8383,  ...,  0.3760,  1.5772,  0.5448],
         [ 0.2969, -0.2529, -0.3323,  ..., -1.1244,  1.0921, -1.5117],
         [ 0.0225,  0.7831, -0.4103,  ...,  0.8218, -0.7692,  0.5307]],

        [[-0.9830, -0.4152,  0.1385,  ..., -1.0654,  1.8724, -1.0763],
         [ 1.8685, -0.2894,  0.7087,  ..., -0

Test your transformer implementation here

In [246]:
def test_transformer():
    num_tokens = 100
    batch_size = 10
    dim = 64
    num_layers = 4
    num_heads = 2
    dummy_model = Transformer(
        dim=dim, attn_dim=32, mlp_dim=dim, num_heads=num_heads, num_layers=num_layers
    )

    inp = torch.randn(batch_size, num_tokens, dim)
    print(f"Input: {inp.shape=}")
    # test case 1 regular forward pass
    print("Test Case 1")
    with torch.no_grad():
        output, alpha = dummy_model(inp, attn_mask=None)
        # assert alpha is None
        assert output.shape == (
            batch_size,
            num_tokens,
            dim,
        ), f"wrong output shape {output.shape}"
    print("Passed")
    # test case 2 collect attentions
    print("Test Case 2")
    with torch.no_grad():
        output, alpha = dummy_model(inp, attn_mask=None, return_attn=True)
        assert output.shape == (
            batch_size,
            num_tokens,
            dim,
        ), f"wrong output shape {output.shape}"
        print("Test case 2 output correct")
        assert alpha.shape == (
            batch_size,
            num_layers,
            num_heads,
            num_tokens,
            num_tokens,
        ), f"wrong alpha shape {alpha.shape}"

    print("Test Case 3")
    # test case 3 with attention mask
    attn_mask = torch.zeros(batch_size, num_tokens, num_tokens)
    attn_mask[:, torch.arange(num_tokens), torch.arange(num_tokens)] = 1
    attn_mask[:, torch.arange(num_tokens)[1:], torch.arange(num_tokens)[:-1]] = 1
    with torch.no_grad():
        output, alpha = dummy_model(inp, attn_mask=attn_mask, return_attn=True)
        print("Attention mask pattern", attn_mask[0])
        print("Alpha pattern", alpha[0, 0, 0])
        assert torch.all(alpha.permute(1, 2, 0, 3, 4)[:, :, attn_mask == 0] == 0).item()
    print("Test case 3 passed!\n\n")
    print("Test Case 4")
    # test case 4 creates a causal mask where each token can only attend to previous tokens and itself
    causal_mask = (
        torch.tril(torch.ones(num_tokens, num_tokens))
        .unsqueeze(0)
        .repeat(batch_size, 1, 1)
    )  # Shape: (B, T, T)

    with torch.no_grad():
        output, alpha = dummy_model(inp, attn_mask=causal_mask, return_attn=True)
        # Verify the causal mask
        for b in range(batch_size):
            for l in range(num_layers):
                for h in range(num_heads):
                    attn_weights = alpha[b, l, h]  # Shape: (T, T)
                    # Positions where j > i should have zero attention weights
                    # We can create a boolean mask for j > i
                    future_mask = torch.triu(
                        torch.ones(num_tokens, num_tokens), diagonal=1
                    ).bool()  # Shape: (T, T)
                    # Extract attention weights for future positions
                    future_attn = attn_weights[future_mask]
                    # Assert that these weights are close to zero
                    assert torch.all(
                        future_attn < 1e-6
                    ), f"Causal mask violated in batch {b}, layer {l}, head {h}"

    torch.save(dummy_model, "transformer_mask.pth")


test_transformer()

6240
6240
6240
6240
6240
6240
6240
6240
Input: inp.shape=torch.Size([10, 100, 64])
Test Case 1


-------------------------Transformer: torch.Size([10, 100, 64])


--------- Attention Residual


 In Multi headed Attention: x.shape=torch.Size([10, 100, 64])


-----------------------
In Attention Head:

=
x.shape=torch.Size([10, 100, 64])
Q.shape=torch.Size([10, 100, 32]),K.shape=torch.Size([10, 100, 32]),V.shape=torch.Size([10, 100, 32])
A.shape=torch.Size([10, 100, 100])
Second way A.shape=torch.Size([10, 100, 100])
Multi Headed Attention: out.shape=torch.Size([10, 100, 32]),alpha.shape=torch.Size([10, 100, 100])


-----------------------
In Attention Head:

=
x.shape=torch.Size([10, 100, 64])
Q.shape=torch.Size([10, 100, 32]),K.shape=torch.Size([10, 100, 32]),V.shape=torch.Size([10, 100, 32])
A.shape=torch.Size([10, 100, 100])
Second way A.shape=torch.Size([10, 100, 100])
Multi Headed Attention: out.shape=torch.Size([10, 100, 32]),alpha.shape=torch.Size([10, 100, 100])
10,10
Multi Head

In [234]:
num_tokens = 100
batch_size = 10
dim = 64
num_layers = 4
num_heads = 2
dummy_model = Transformer(
    dim=dim, attn_dim=32, mlp_dim=dim, num_heads=num_heads, num_layers=num_layers
).to(device)

inp = torch.randn(batch_size, num_tokens, dim).to(device)
print(f"Input: {inp.shape=}")
# test case 1 regular forward pass
print("Test Case 1")
with torch.no_grad():
    output, alpha = dummy_model(inp, attn_mask=None)
    if len(output) == 0:
        output = None
torch.save(dummy_model, "model.pth")

6240
6240
6240
6240
6240
6240
6240
6240
Input: inp.shape=torch.Size([10, 100, 64])
Test Case 1


-------------------------Transformer: torch.Size([10, 100, 64])


--------- Attention Residual


 In Multi headed Attention: x.shape=torch.Size([10, 100, 64])


-----------------------
In Attention Head:

=
x.shape=torch.Size([10, 100, 64])
Q.shape=torch.Size([10, 100, 32]),K.shape=torch.Size([10, 100, 32]),V.shape=torch.Size([10, 100, 32])
A.shape=torch.Size([10, 100, 100])
Second way A.shape=torch.Size([10, 100, 100])
Multi Headed Attention: out.shape=torch.Size([10, 100, 32]),alpha.shape=torch.Size([10, 100, 100])


-----------------------
In Attention Head:

=
x.shape=torch.Size([10, 100, 64])
Q.shape=torch.Size([10, 100, 32]),K.shape=torch.Size([10, 100, 32]),V.shape=torch.Size([10, 100, 32])
A.shape=torch.Size([10, 100, 100])
Second way A.shape=torch.Size([10, 100, 100])
Multi Headed Attention: out.shape=torch.Size([10, 100, 32]),alpha.shape=torch.Size([10, 100, 100])
10,10
Multi Head

In [363]:
print(alpha.shape)

torch.Size([10, 100, 64])


In [364]:
output

## Test Transformer

In [235]:
def test_transformer():
    num_tokens = 100
    batch_size = 10
    dim = 64
    num_layers = 4
    num_heads = 2
    dummy_model = Transformer(
        dim=dim, attn_dim=32, mlp_dim=dim, num_heads=num_heads, num_layers=num_layers
    ).to(device)

    inp = torch.randn(batch_size, num_tokens, dim).to(device)
    print(f"Input: {inp.shape=}")
    # test case 1 regular forward pass
    print("Test Case 1")
    with torch.no_grad():
        output, alpha = dummy_model(inp, attn_mask=None)
        # print(f"\n\n\n{output.shape=},{alpha.shape=}")

        # if len(alpha)==0:
        #     alpha=None
        assert alpha is None
        assert output.shape == (
            batch_size,
            num_tokens,
            dim,
        ), f"wrong output shape {output.shape}"

    # test case 2 collect attentions


test_transformer()

6240
6240
6240
6240
6240
6240
6240
6240
Input: inp.shape=torch.Size([10, 100, 64])
Test Case 1


-------------------------Transformer: torch.Size([10, 100, 64])


--------- Attention Residual


 In Multi headed Attention: x.shape=torch.Size([10, 100, 64])


-----------------------
In Attention Head:

=
x.shape=torch.Size([10, 100, 64])
Q.shape=torch.Size([10, 100, 32]),K.shape=torch.Size([10, 100, 32]),V.shape=torch.Size([10, 100, 32])
A.shape=torch.Size([10, 100, 100])
Second way A.shape=torch.Size([10, 100, 100])
Multi Headed Attention: out.shape=torch.Size([10, 100, 32]),alpha.shape=torch.Size([10, 100, 100])


-----------------------
In Attention Head:

=
x.shape=torch.Size([10, 100, 64])
Q.shape=torch.Size([10, 100, 32]),K.shape=torch.Size([10, 100, 32]),V.shape=torch.Size([10, 100, 32])
A.shape=torch.Size([10, 100, 100])
Second way A.shape=torch.Size([10, 100, 100])
Multi Headed Attention: out.shape=torch.Size([10, 100, 32]),alpha.shape=torch.Size([10, 100, 100])
10,10
Multi Head

In [250]:
def test_transformer_2():
    num_tokens = 100
    batch_size = 10
    dim = 64
    num_layers = 4
    num_heads = 2
    dummy_model = Transformer(
        dim=dim, attn_dim=32, mlp_dim=dim, num_heads=num_heads, num_layers=num_layers
    ).to(device)

    inp = torch.randn(batch_size, num_tokens, dim).to(device)
    print(f"Input: {inp.shape=}")
    print("Test Case 2")
    with torch.no_grad():
        output, alpha = dummy_model(inp, attn_mask=None, return_attn=True)
        assert output.shape == (
            batch_size,
            num_tokens,
            dim,
        ), f"wrong output shape {output.shape}"
        assert alpha.shape == (
            batch_size,
            num_layers,
            num_heads,
            num_tokens,
            num_tokens,
        ), f"wrong alpha shape {alpha.shape}"

    print("Test Case 3")
    # test case 3 with attention mask
    attn_mask = torch.zeros(batch_size, num_tokens, num_tokens).to(device)
    attn_mask[:, torch.arange(num_tokens), torch.arange(num_tokens)] = 1
    attn_mask[:, torch.arange(num_tokens)[1:], torch.arange(num_tokens)[:-1]] = 1
    attn_mask = attn_mask.to(device)
    with torch.no_grad():
        output, alpha = dummy_model(inp, attn_mask=attn_mask, return_attn=True)
        print("Attention mask pattern", attn_mask[0])
        print("Alpha pattern", alpha[0, 0, 0])
        assert torch.all(alpha.permute(1, 2, 0, 3, 4)[:, :, attn_mask == 0] == 0).item()

    print("Test Case 4")
    # test case 4 creates a causal mask where each token can only attend to previous tokens and itself
    causal_mask = (
        torch.tril(torch.ones(num_tokens, num_tokens))
        .unsqueeze(0)
        .repeat(batch_size, 1, 1)
    ).to(
        device
    )  # Shape: (B, T, T)

    with torch.no_grad():
        output, alpha = dummy_model(inp, attn_mask=causal_mask, return_attn=True)
        # Verify the causal mask
        i = 0
        future_mask = None
        for b in range(batch_size):
            for l in range(num_layers):
                for h in range(num_heads):
                    attn_weights = alpha[b, l, h]  # Shape: (T, T)
                    # Positions where j > i should have zero attention weights
                    # We can create a boolean mask for j > i
                    future_mask = torch.triu(
                        torch.ones(num_tokens, num_tokens), diagonal=1
                    ).bool()  # Shape: (T, T)
                    # if i==0:
                    #     print(future_mask)
                    # Extract attention weights for future positions
                    future_attn = attn_weights[future_mask]
                    # Assert that these weights are close to zero
                    assert torch.all(
                        future_attn < 1e-6
                    ), f"Causal mask violated in batch {b}, layer {l}, head {h}"

        print(future_mask)


test_transformer_2()

6240
6240
6240
6240
6240
6240
6240
6240
Input: inp.shape=torch.Size([10, 100, 64])
Test Case 2


-------------------------Transformer: torch.Size([10, 100, 64])


--------- Attention Residual


 In Multi headed Attention: x.shape=torch.Size([10, 100, 64])


-----------------------
In Attention Head:

=
x.shape=torch.Size([10, 100, 64])
Q.shape=torch.Size([10, 100, 32]),K.shape=torch.Size([10, 100, 32]),V.shape=torch.Size([10, 100, 32])
A.shape=torch.Size([10, 100, 100])
Second way A.shape=torch.Size([10, 100, 100])
Multi Headed Attention: out.shape=torch.Size([10, 100, 32]),alpha.shape=torch.Size([10, 100, 100])


-----------------------
In Attention Head:

=
x.shape=torch.Size([10, 100, 64])
Q.shape=torch.Size([10, 100, 32]),K.shape=torch.Size([10, 100, 32]),V.shape=torch.Size([10, 100, 32])
A.shape=torch.Size([10, 100, 100])
Second way A.shape=torch.Size([10, 100, 100])
Multi Headed Attention: out.shape=torch.Size([10, 100, 32]),alpha.shape=torch.Size([10, 100, 100])
10,10
Multi Head

In [266]:
dummy_model = Transformer(dim, attn_dim, dim, 4, 10).to(device)
print("Test Case 4")
# test case 4 creates a causal mask where each token can only attend to previous tokens and itself
causal_mask = (
    torch.tril(torch.ones(num_tokens, num_tokens)).unsqueeze(0).repeat(batch_size, 1, 1)
).to(
    device
)  # Shape: (B, T, T)

with torch.no_grad():
    output, alpha = dummy_model(inp, attn_mask=causal_mask, return_attn=True)
    # Verify the causal mask
    i = 0
    future_mask = None
    for b in range(batch_size):
        for l in range(num_layers):
            for h in range(num_heads):
                attn_weights = alpha[b, l, h]  # Shape: (T, T)
                # Positions where j > i should have zero attention weights
                # We can create a boolean mask for j > i
                future_mask = torch.triu(
                    torch.ones(num_tokens, num_tokens), diagonal=1
                ).bool()  # Shape: (T, T)
                # if i==0:
                #     print(future_mask)
                # Extract attention weights for future positions
                future_attn = attn_weights[future_mask]
                # Assert that these weights are close to zero
                assert torch.all(
                    future_attn < 1e-6
                ), f"Causal mask violated in batch {b}, layer {l}, head {h}"

    # print(future_mask)
    print(future_attn)

6240
6240
6240
6240
6240
6240
6240
6240
6240
6240
6240
6240
6240
6240
6240
6240
6240
6240
6240
6240
6240
6240
6240
6240
6240
6240
6240
6240
6240
6240
6240
6240
6240
6240
6240
6240
6240
6240
6240
6240
Test Case 4


-------------------------Transformer: torch.Size([10, 100, 64])


 In Multi headed Attention: x.shape=torch.Size([10, 100, 64])


-----------------------
In Attention Head:

=
x.shape=torch.Size([10, 100, 64])
Multi Headed Attention: out.shape=torch.Size([10, 100, 32]),alpha.shape=torch.Size([10, 100, 100])


-----------------------
In Attention Head:

=
x.shape=torch.Size([10, 100, 64])
Multi Headed Attention: out.shape=torch.Size([10, 100, 32]),alpha.shape=torch.Size([10, 100, 100])


-----------------------
In Attention Head:

=
x.shape=torch.Size([10, 100, 64])
Multi Headed Attention: out.shape=torch.Size([10, 100, 32]),alpha.shape=torch.Size([10, 100, 100])


-----------------------
In Attention Head:

=
x.shape=torch.Size([10, 100, 64])
Multi Headed Attention: out.shape

In [253]:
torch.save(dummy_model, "final_transformer.pth")

In [267]:
print("Test Case 3")
# test case 3 with attention mask
attn_mask = torch.zeros(batch_size, num_tokens, num_tokens).to(device)
attn_mask[:, torch.arange(num_tokens), torch.arange(num_tokens)] = 1
attn_mask[:, torch.arange(num_tokens)[1:], torch.arange(num_tokens)[:-1]] = 1
attn_mask = attn_mask.to(device)
with torch.no_grad():
    output, alpha = dummy_model(inp, attn_mask=attn_mask, return_attn=True)
    print("Attention mask pattern", attn_mask[0])
    print("Alpha pattern", alpha[0, 0, 0])
    assert torch.all(alpha.permute(1, 2, 0, 3, 4)[:, :, attn_mask == 0] == 0).item()


Test Case 3


-------------------------Transformer: torch.Size([10, 100, 64])


 In Multi headed Attention: x.shape=torch.Size([10, 100, 64])


-----------------------
In Attention Head:

=
x.shape=torch.Size([10, 100, 64])
Multi Headed Attention: out.shape=torch.Size([10, 100, 32]),alpha.shape=torch.Size([10, 100, 100])


-----------------------
In Attention Head:

=
x.shape=torch.Size([10, 100, 64])
Multi Headed Attention: out.shape=torch.Size([10, 100, 32]),alpha.shape=torch.Size([10, 100, 100])


-----------------------
In Attention Head:

=
x.shape=torch.Size([10, 100, 64])
Multi Headed Attention: out.shape=torch.Size([10, 100, 32]),alpha.shape=torch.Size([10, 100, 100])


-----------------------
In Attention Head:

=
x.shape=torch.Size([10, 100, 64])
Multi Headed Attention: out.shape=torch.Size([10, 100, 32]),alpha.shape=torch.Size([10, 100, 100])
attn_alphas.shape=torch.Size([10, 4, 100, 100])
Multi Headed Attention Successful


--------------------------
x.shape=torch.Size([10,

In [271]:
print(alpha[0,1,0])

tensor([[1.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
        [0.6469, 0.3531, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
        [0.0000, 0.5915, 0.4085,  ..., 0.0000, 0.0000, 0.0000],
        ...,
        [0.0000, 0.0000, 0.0000,  ..., 0.4090, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000,  ..., 0.5537, 0.4463, 0.0000],
        [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.4433, 0.5567]],
       device='mps:0')


### Debug

In [221]:
w = torch.randn(32, 200, 100)

torch.Size([100, 200, 32])

In [226]:
20000 / 32

625.0

In [225]:
w.transpose(dim0=0, dim1=1).shape

torch.Size([200, 32, 100])

In [240]:
Q = torch.randn(32, 100, 10)
K = torch.randn(32, 100, 10)
(Q @ K.transpose(-2, -1)).shape

torch.Size([32, 100, 100])

In [286]:
Q1 = torch.tensor([[1],[2]])
V1 = copy.deepcopy(Q1)
V1_T=V1.transpose(0,-1)
V1_T

tensor([[1, 2]])

In [285]:
Q1@V1_T

tensor([[1, 2],
        [2, 4]])

## Tokenization

Transformers are typically designed to handle discrete token sequences, like
words in a sentence. However, in many domains like audio and images, the input is
continuous rather than discrete. Explain
how the transformer architecture can be adapted to handle continuous inputs, such
as audio and images. Think about how tokenization, embedding, and positional
encodings need to be adjusted to effectively apply self-attention to these types of
inputs. Consider how you would divide continuous data into meaningful ”tokens”
and how the transformer can capture local and global relationships within this data.

## Problem 3: Vision Transformer

## Part 3.A

In [ ]:
class PatchEmbed(nn.Module):
    """Image to Patch Embedding"""

    def __init__(self, img_size: int, patch_size: int, nin: int, nout: int):
        # img_size       the width and height of the image. you can assume that
        #                the images will be square
        # patch_size     the width of each square patch. You can assume that
        #                img_size is divisible by patch_size
        # nin            the number of input channels
        # nout           the number of output channels

        super().__init__()
        assert img_size % patch_size == 0

        self.img_size = img_size
        self.num_patches = (img_size // patch_size) ** 2

        # TODO Set up parameters for the Patch Embedding
        # ======= Answer START ========

        # ======= Answer END ========

    def forward(self, x: torch.Tensor):
        # x        the input image. shape: (B, nin, Height, Width)
        #
        # Output
        # out      the patch embeddings for the input. shape: (B, num_patches, nout)

        # TODO: Implement the patch embedding. You want to split up the image into
        # square patches of the given patch size. Then each patch_size x patch_size
        # square should be linearly projected into an embedding of size nout.
        #
        # Hint: Take a look at nn.Conv2d. How can this be used to perform the
        #       patch embedding?
        out = None

        # ======= Answer START ========

        # ======= Answer END ========

        return out

## Part 3.B

In [ ]:
class VisionTransformer(nn.Module):
    def __init__(
        self,
        n_channels: int,
        nout: int,
        img_size: int,
        patch_size: int,
        dim: int,
        attn_dim: int,
        mlp_dim: int,
        num_heads: int,
        num_layers: int,
    ):
        # n_channels       number of input image channels
        # nout             desired output dimension
        # img_size         width of the square image
        # patch_size       width of the square patch
        # dim              embedding dimension
        # attn_dim         the hidden dimension of the attention layer
        # mlp_dim          the hidden layer dimension of the FFN
        # num_heads        the number of heads in the attention layer
        # num_layers       the number of attention layers.
        super().__init__()
        self.patch_embed = PatchEmbed(
            img_size=img_size, patch_size=patch_size, nin=n_channels, nout=dim
        )
        self.pos_E = nn.Embedding(
            (img_size // patch_size) ** 2, dim
        )  # positional embedding matrix

        self.cls_token = nn.Parameter(torch.randn(1, 1, dim))  # learned class embedding
        self.transformer = Transformer(
            dim=dim,
            attn_dim=attn_dim,
            mlp_dim=mlp_dim,
            num_heads=num_heads,
            num_layers=num_layers,
        )

        self.head = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, nout))

    def forward(
        self, img: torch.Tensor, return_attn=False
    ) -> Tuple[torch.Tensor, Optional[torch.Tensor]]:
        # img          the input image. shape: (B, nin, img_size, img_size)
        # return_attn  whether to return the attention alphas
        #
        # Outputs
        # out          the output of the vision transformer. shape: (B, nout)
        # alphas       the attention weights for all heads and layers. None if return_attn is False, otherwise
        #              shape: (B, num_layers, num_heads, num_patches + 1, num_patches + 1)

        # generate embeddings
        embs = self.patch_embed(img)  # patch embedding
        B, T, _ = embs.shape
        pos_ids = torch.arange(T).expand(B, -1).to(embs.device)
        embs += self.pos_E(pos_ids)  # positional embedding

        cls_token = self.cls_token.expand(len(embs), -1, -1)
        x = torch.cat([cls_token, embs], dim=1)

        x, alphas = self.transformer(x, attn_mask=None, return_attn=return_attn)
        out = self.head(x)[:, 0]
        return out, alphas

## Part 3.C

In [ ]:
# set up the dataset and dataloader

MEAN = [0.4914, 0.4822, 0.4465]
STD = [0.2470, 0.2435, 0.2616]
img_transform = transforms.Compose(
    [
        transforms.ToTensor(),
        transforms.Normalize(mean=MEAN, std=STD),
    ]
)
inv_transform = transforms.Compose(
    [
        transforms.Normalize(mean=[0.0, 0.0, 0.0], std=1 / np.array(STD)),
        transforms.Normalize(mean=-np.array(MEAN), std=[1.0, 1.0, 1.0]),
        transforms.ToPILImage(),
    ]
)


train_dataset = torchvision.datasets.CIFAR10(
    train=True, root="data", transform=img_transform, download=True
)
val_dataset = torchvision.datasets.CIFAR10(
    train=False, root="data", transform=img_transform
)
train_dataloader = torch.utils.data.DataLoader(
    train_dataset, batch_size=256, shuffle=True, num_workers=10
)
val_dataloader = torch.utils.data.DataLoader(
    val_dataset, batch_size=256, shuffle=False, num_workers=10
)

In [ ]:
# set up the model and optimizer

import torch.optim as optim

model = VisionTransformer(
    n_channels=3,
    nout=10,
    img_size=32,
    patch_size=4,
    dim=128,
    attn_dim=64,
    mlp_dim=128,
    num_heads=3,
    num_layers=6,
).cuda()

criterion = nn.CrossEntropyLoss()


NUM_EPOCHS = 10
optimizer = optim.AdamW(model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

In [ ]:
# evaluate the model
def evaluate_cifar_model(model, criterion, val_loader):
    is_train = model.training
    model.eval()
    with torch.no_grad():
        loss_meter, acc_meter = AverageMeter(), AverageMeter()
        for img, labels in val_loader:
            # move all img, labels to device (cuda)
            img = img.cuda()
            labels = labels.cuda()
            outputs, _ = model(img)
            loss_meter.update(criterion(outputs, labels).item(), len(img))
            acc = (outputs.argmax(-1) == labels).float().mean().item()
            acc_meter.update(acc, len(img))
    model.train(is_train)
    return loss_meter.calculate(), acc_meter.calculate()

In [ ]:
# Time Estimate: less than 5 minutes on T4 GPU
# train the model
import tqdm

for epoch in range(NUM_EPOCHS):  #
    loss_meter = AverageMeter()
    acc_meter = AverageMeter()
    for img, labels in tqdm.tqdm(train_dataloader):
        img, labels = img.cuda(), labels.cuda()

        optimizer.zero_grad()

        outputs, _ = model(img)
        loss = criterion(outputs, labels)
        loss_meter.update(loss.item(), len(img))
        acc = (outputs.argmax(-1) == labels).float().mean().item()
        acc_meter.update(acc, len(img))
        loss.backward()
        optimizer.step()
    scheduler.step()
    print(
        f"Train Epoch: {epoch}, Loss: {loss_meter.calculate()}, Acc: {acc_meter.calculate()}"
    )
    if epoch % 10 == 0:
        val_loss, val_acc = evaluate_cifar_model(model, criterion, val_dataloader)
        print(f"Val Epoch: {epoch}, Loss: {val_loss}, Acc: {val_acc}")

val_loss, val_acc = evaluate_cifar_model(model, criterion, val_dataloader)
print(f"Val Epoch: {epoch}, Loss: {val_loss}, Acc: {val_acc}")
print("Finished Training")

# Part 3.D

In [ ]:
for val_batch in val_dataloader:
    break

model.eval()
with torch.no_grad():
    img, labels = val_batch
    img = img.cuda()
    outputs, attns = model(img, return_attn=True)

fig, ax = plt.subplots(2, 10, figsize=(10, 2))
for i in range(10):
    flattened_attns = (
        attns.flatten(1, 2)[:, :, 0, 1:].mean(1).reshape(-1, 8, 8).cpu().numpy()
    )
    ax[0, i].imshow(inv_transform(img[i]))
    ax[1, i].imshow(flattened_attns[i])
    ax[0, i].axis(False)
    ax[1, i].axis(False)

# Problem 4: Dialogue GPT

In [ ]:
!pip install wget

In [ ]:
import wget
import os

if not os.path.exists("input.txt"):
    wget.download(
        "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
    )

In [ ]:
with open("input.txt", "r") as f:
    raw_text = f.read()
all_dialogues = raw_text.split("\n\n")

In [ ]:
import nltk
from nltk.tokenize import word_tokenize

nltk.download("punkt")

## Part 4.A

In [ ]:
def tokenize(s):
    return word_tokenize(s)


class MyTokenizer:
    def __init__(self, raw_text: str):
        # raw_text     contains the text from which we will build our vocabulary

        self.start = "<START>"  # token that starts every example
        self.pad = "<PAD>"  # token used to pad examples to the same length
        self.unk = "<UNK>"  # token used if encountering a word not in our vocabulary

        vocab = np.unique(tokenize(raw_text))
        vocab = np.concatenate([np.array([self.start, self.pad, self.unk]), vocab])

        self.vocab = vocab  # array of tokens in order
        self.tok_to_id = {w: i for i, w in enumerate(vocab)}  # mapping of token to ID
        self.vocab_size = len(self.vocab)  # size of vocabulary

    def encode(self, s: str) -> torch.Tensor:
        # s           input string
        #
        # Output
        # id_tensor   a tensor of token ids, starting with the start token.t

        id_tensor = None

        # TODO: tokenize the input using word_tokenize. Return a tensor
        # of the token ids, starting with the token id for the start token.
        # ============ ANSWER START ===========

        # ============ ANSWER END =============

        return id_tensor

    def decode(self, toks: torch.Tensor) -> str:
        # toks         a list of token ids
        #
        # Output
        # decoded_str  the token ids decoded back into a string (join with a space)

        decoded_str = None

        # TODO: convert the token ids back to the actual corresponding words.
        # Join the tokens with a space and return the full string
        # ============ ANSWER START ===========

        # ============ ANSWER END =============

        return decoded_str

    def pad_examples(self, tok_list: List[torch.Tensor]) -> torch.Tensor:
        # Pads the tensors to the right with the pad token so that they are the same length.
        #
        # tok_list       a list of tensors containing token ids (maybe of different lengths)
        #
        # Output
        # padded_tokens  shape: (len(tok_list), max length within tok_list)
        return torch.nn.utils.rnn.pad_sequence(
            tok_list, batch_first=True, padding_value=self.tok_to_id[self.pad]
        )


tok = MyTokenizer(raw_text)

In [ ]:
# tokenizer test cases
input_string = "KING RICHARD III:\nSay that I did all this for love of her."
enc = tok.encode(input_string)
print(enc)
dec = tok.decode(enc)
print(dec)
assert dec == "<START> KING RICHARD III : Say that I did all this for love of her ."

# Part 4.B

In [ ]:
class DialogueDataset:
    def __init__(self, tokenizer: MyTokenizer, lines: List[str], max_N: int):
        # tokenizer    an instance of MyTokenizer
        # lines        a list of strings. each element in an example in the dataset
        # max_N        the maximum number of tokens allowed per example. More than this will be truncated
        self.lines = lines
        self.tokenizer = tokenizer
        self.max_N = max_N

    def __len__(self) -> int:
        return len(self.lines)

    def __getitem__(self, idx: int) -> torch.Tensor:
        # returns the example at int encoded by the tokenizer
        # truncates the example if it is more than max_N tokens
        return self.tokenizer.encode(self.lines[idx])[: self.max_N]


def collate_fn(examples: List[torch.Tensor]):
    # examples        a batch of tensors containing token ids (maybe of different lengths)
    # Outputs a dictionary containing
    #   input_ids     a single tensor with all of the examples padded (from the right) to the max
    #                 length within the batch. shape:(B, max length within examples)
    #   input_mask    a tensor indicating which tokens are padding and should be ignored. 0 if padding
    #                 and 1 if not. shape: (B, max length within examples)
    new_input_ids = tok.pad_examples(examples)
    attn_mask = torch.ones(new_input_ids.shape)
    attn_mask[new_input_ids == tok.tok_to_id[tok.pad]] = 0
    return {"input_ids": tok.pad_examples(examples), "input_mask": attn_mask}


ds = DialogueDataset(tok, all_dialogues, max_N=200)
training_dl = torch.utils.data.DataLoader(ds, batch_size=64, collate_fn=collate_fn)

In [ ]:
# take a look at an example of an element from the training dataloader
for batch in training_dl:
    print(batch)
    break

## Part 4.C

In [ ]:
embs = torch.ones((32, 100, 128))
B, T, _ = embs.shape
pos_ids = torch.arange(T).expand(B, -1)
print(pos_ids)
pos_E = nn.Embedding(200, 128)
print(pos_E)
pos_E(pos_ids).shape

In [ ]:
class DialogueGPT(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        max_N: int,
        dim: int,
        attn_dim: int,
        mlp_dim: int,
        num_heads: int,
        num_layers: int,
    ):
        # vocab_size       size of the vocabulary
        # max_N            maximum number of tokens allowed to appear in 1 example
        # dim              embedding dimension
        # attn_dim         the hidden dimension of the attention layer
        # mlp_dim          the hidden layer dimension of the FFN
        # num_heads        the number of heads in the attention layer
        # num_layers       the number of attention layers.

        super().__init__()

        # TODO: set up the token embedding and positional embeddings
        #       Hint, use nn.Embedding
        # ============ ANSWER START ============

        # ============ ANSWER END ==============

        self.transformer = Transformer(
            dim=dim,
            attn_dim=attn_dim,
            mlp_dim=mlp_dim,
            num_heads=num_heads,
            num_layers=num_layers,
        )

        self.head = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, vocab_size))

    def forward(
        self, input_ids: torch.Tensor, return_attn=False
    ) -> Tuple[torch.Tensor, Optional[torch.Tensor]]:
        # input_ids     a batch of input ids (right padded). shape: (B x T)
        # return_attn   whether to return the attention weights
        #
        # Output
        # out           the logit vector (B x T x V)
        # alphas        the attention weights if return_attn is True. Otherwise None shape: (B, num_layers, num_heads, T, T)

        embs = None

        # TODO: retrieve the token embeddings for the input_ids.
        #       Add to the token embeddings the positional embeddings.
        #       Store the combined embedding in embs
        # ============ ANSWER START ============

        # ============ ANSWER END ============

        causal_attn_mask = None

        # TODO: Create the causal attention mask, which should be of size (B, T, T)
        #       Remember that the causal attention mask is lower triangular (all tokens only
        #       depend on themselves and the tokens before them).
        # .      Store the mask in causal_attn_mask
        # Hint: check out torch.tril
        # ============ ANSWER START ============

        # ============ ANSWER END ==============

        x, alphas = self.transformer(
            embs, attn_mask=causal_attn_mask, return_attn=return_attn
        )
        out = self.head(x)
        return out, alphas

    def generate(self, input_ids, num_tokens):
        # you can assume batch size 1
        with torch.no_grad():
            for i in range(num_tokens):
                out, _ = self.forward(input_ids)
                new_token = torch.argmax(out[:, [-1]], -1)
                input_ids = torch.cat([input_ids, new_token], dim=1)
        return input_ids

## Part 4.D

In [ ]:
class DialogueLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.criterion = nn.CrossEntropyLoss(reduction="none")

    def forward(
        self, logits: torch.Tensor, input_ids: torch.Tensor, inp_mask: torch.Tensor
    ):
        # logits      the logits produced by DialogueGPT. shape: (B x T x V)
        # input_ids   the token ids. shape: (B x T)
        # inp_mask    a 0/1 mask of which tokens are padding tokens and should be ignored. shape: (B x T)

        # TODO: Implement the language model loss. For logits[i], we want to supervise the i+1 token_id
        # with the cross entropy loss. We thus will not supervise the start token (input_ids[0]) or use
        # the last logit vector (logits[-1]). Return the average of the losses for each token in the batch,
        # making sure to ignore tokens corresponding to the padding (use inp_mask).

        # ============ ANSWER START ============

        # ============ ANSWER END ==============
        return loss

## Part 4.F

In [ ]:
import torch.optim as optim

model = DialogueGPT(
    vocab_size=tok.vocab_size,
    max_N=200,
    dim=128,
    attn_dim=64,
    mlp_dim=128,
    num_heads=3,
    num_layers=6,
).cuda()
criterion = DialogueLoss()

NUM_EPOCHS = 80


optimizer = optim.AdamW(
    model.parameters(), lr=0.0001, weight_decay=0
)  # implement in homework
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

In [ ]:
# Time estimate: around 30 minutes on T4 GPU
# Training
import tqdm

for epoch in range(NUM_EPOCHS):  # loop over the dataset multiple times
    loss_meter = AverageMeter()
    for inp_dict in tqdm.tqdm(training_dl):
        # get the inputs; data is a list of [inputs, labels]
        inp_ids, inp_mask = inp_dict["input_ids"], inp_dict["input_mask"]
        inp_ids = inp_ids.cuda()
        inp_mask = inp_mask.cuda()
        # zero the parameter gradients
        optimizer.zero_grad()

        # forward + backward + optimize
        outputs, _ = model(input_ids=inp_ids)
        loss = criterion(outputs, inp_ids, inp_mask)
        loss_meter.update(loss.item(), len(inp_dict["input_ids"]))
        loss.backward()
        optimizer.step()
    scheduler.step()

    # print example
    inp = tok.encode("").unsqueeze(0).cuda()
    print(tok.decode(model.generate(inp, 10)[0].cpu()))

    print(
        f"Train Epoch: {epoch}, Loss: {loss_meter.calculate():0.4f}, LR: {scheduler.get_last_lr()[0]}"
    )

## Part 4.G

In [ ]:
inp = tok.encode("").unsqueeze(0).cuda()
print(tok.decode(model.generate(inp, 50)[0].cpu()))